# MiniGPT From Scratch：手写 Decoder-only Transformer

本 Notebook 沿用原来的模块和实现思路，
按从输入到输出的模型路径组织。每个模块只定义一次，按顺序执行即可独立运行。

使用正弦位置编码、独立 Q/K/V 投影、ReLU FFN 和 Post-LayerNorm。
这是学习用的简化 GPT，不是某个 GPT 论文模型的逐项复现，也与 MiniGPT-4 无关。
示例使用合成 token ids，没有文本 tokenizer，生成结果也是 token ids。

形状记号：`B` 为 batch size，`T` 为序列长度，`D` 为 embedding 维度，
`H` 为头数，`Dh = D / H` 为每个头的维度，`V` 为词表大小。
本例 `D=32`（偶数）、`H=4`，位置编码长度上限为 20。

In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

## 1. Token Embedding

输入 `(B, T)` 的 token ids，经 `nn.Embedding(V, D)` 查表后，
输出 `(B, T, D)` 的 hidden states。这个独立例子先观察形状；
第 7 节的 MiniGPT 会在模型内部创建自己的 embedding 层。

In [2]:
embedding = nn.Embedding(100, 32)
token_ids = torch.tensor([[10, 20, 30], [20, 30, 40]])
hidden_states = embedding(token_ids)
print("token ids:", token_ids.shape)
print("hidden states:", hidden_states.shape)

token ids: torch.Size([2, 3])
hidden states: torch.Size([2, 3, 32])


## 2. Positional Encoding

Self-attention 本身不会显式标记序列顺序，因此需要将位置信息加入输入。
保留原来的正弦/余弦位置编码：偶数特征维用 sin，奇数特征维用 cos，
不同维度对应不同频率。

$$PE(pos,2i)=\sin\left(pos/10000^{2i/D}\right)$$
$$PE(pos,2i+1)=\cos\left(pos/10000^{2i/D}\right)$$

输入与输出均为 `(B, T, D)`；缓存 `pe` 为 `(1, max_len, D)`，
取前 `T` 个位置后通过广播与每个样本相加。
`register_buffer` 保存固定编码，不把它当作可训练参数。
这份实现使用偶数 `embed_dim`，直接 forward 的序列长度不能超过 `max_len`。

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            raise ValueError("序列长度超过位置编码的 max_len")
        x = x + self.pe[:, :seq_len]
        return x

## 3. Multi-head Self Attention

保留 `MaskedMultiHeadAttention`：Q、K、V 来自同一输入，多个头分别计算注意力。
`D` 必须能被 `H` 整除；每个头观察维度为 `Dh` 的投影特征。

### QKV Projection
`W_q`、`W_k`、`W_v` 是三个可学习的 Linear，分别输出 Q/K/V。
输入 `(B, T, D)` → 每个输出 `(B, T, D)`。

### Split Heads
先 reshape，再交换序列轴与头轴：
`(B, T, D)` → `(B, T, H, Dh)` → `(B, H, T, Dh)`。

### Attention Score
K 转置为 `(B, H, Dh, T)`；`Q @ Kᵀ / sqrt(Dh)` 输出 `(B, H, T, T)`。
最后两轴分别表示 Query 位置和 Key 位置。

### Mask 机制
因果遮罩为 `(T, T)` 的下三角矩阵，广播到 `(B, H, T, T)`。
第 `i` 个位置只看 `j ≤ i` 的 token；未来位置的 score 填为负无穷。
输入与输出 score 的 shape 都是 `(B, H, T, T)`。
这样训练时可以并行计算所有位置，又不会偷看 target。
此处只有 causal mask，示例没有 padding，不涉及 padding mask。

### Softmax
沿最后一维（Key 轴）归一化，未来位置权重为 0。
输入 score `(B, H, T, T)` → 输出 weights `(B, H, T, T)`。

### Weighted Value
`weights @ V`：`(B, H, T, T) @ (B, H, T, Dh)` → `(B, H, T, Dh)`。
每个 Query 得到对可见 Value 的加权汇总。

### Combine Heads
`(B, H, T, Dh)` → transpose `(B, T, H, Dh)` → reshape `(B, T, D)`。
最后的 `W_o` 将合并结果再投影，输出仍为 `(B, T, D)`。

In [4]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.embed_dim % self.num_heads == 0

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def split_heads(self, x):
        batch_size, seq_len, embed_dim = x.size()
        x = x.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, head_dim = x.size()
        x = x.transpose(1, 2)
        x = x.reshape(batch_size, seq_len, self.embed_dim)
        return x

    def forward(self, x):
        # (B, T, D) → 三组 (B, T, D) 投影
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 分头：(B, T, D) → (B, H, T, Dh)
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 每个头独立计算 score：(B, H, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        seq_len = x.size(1)
        # 下三角允许关注自身及过去位置；广播到所有样本和头。
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))
        # 沿 Key 轴归一化，未来位置权重为 0。
        attention_weights = torch.softmax(scores, dim = -1)
        # 加权 Value：(B, H, T, T) @ (B, H, T, Dh)
        output = torch.matmul(attention_weights, V)
        # 合头并输出投影：(B, H, T, Dh) → (B, T, D)
        output = self.combine_heads(output)
        output = self.W_o(output)
        return output

## 4. Feed Forward Network

Attention 负责 token 之间的信息交互，FFN 负责每个 token 自身的特征变换。
保留原来的两层 Linear 和 ReLU：`(B, T, D)` → `(B, T, hidden_dim)` → `(B, T, D)`。
所有位置共享同一组 FFN 权重，FFN 不混合不同 token 的位置轴。

In [5]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()

        self.linear1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

## 5. Residual Connection and LayerNorm

Residual 将子层输入与输出相加，使原始信息保留一条直接通路，也有助于梯度传播。
两者 shape 必须同为 `(B, T, D)`。
`nn.LayerNorm(D)` 对每个 token 的最后一个特征维做归一化，再使用可学习的缩放和偏置。

这里沿用原来的 Post-LayerNorm：先相加，再归一化，即 `norm(x + sublayer(x))`。
下一节直接在 DecoderBlock 中组合，不另写一份重复实现。

## 6. Decoder Block

```text
Masked Self Attention
        ↓
Residual + LayerNorm
        ↓
Feed Forward Network
        ↓
Residual + LayerNorm
```

输入和输出均为 `(B, T, D)`，所以可以堆叠多个 Block。
这是 decoder-only 模型，不包含 encoder-decoder cross-attention。

In [6]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = MaskedMultiHeadAttention(embed_dim, num_heads)
        self.ffn = FeedForward(embed_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attention_output = self.attention(x)
        x = self.norm1(x + attention_output)
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)
        return x

## 7. MiniGPT Model

```text
token ids (B, T)
        ↓
Embedding (B, T, D)
        ↓
Positional Encoding (B, T, D)
        ↓
Decoder Blocks (B, T, D)
        ↓
LM Head：Linear(D, V)
        ↓
Vocabulary logits (B, T, V)
```

`ModuleList` 注册每个 DecoderBlock 的参数，forward 依次经过这些层。
每个位置输出整个词表的未归一化 logits；训练时直接交给 CrossEntropyLoss，
生成时才转换为概率。`generate` 与模型一起定义，其运行过程在第 9 节说明。

In [7]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size,embed_dim, num_heads, hidden_dim, num_layers, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position = PositionalEncoding(embed_dim, max_len)
        self.layers = nn.ModuleList([DecoderBlock(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.position(x)
        for layer in self.layers:
            x = layer(x)
        logits = self.lm_head(x)
        return logits

    @torch.no_grad()
    def generate(self, x, max_new_tokens=20, temperature=1.0):
        if temperature <= 0:
            raise ValueError("temperature 必须大于 0")
        for _ in range(max_new_tokens):
            # 输入只保留窗口内的上下文，返回结果仍保留完整序列。
            context = x[:, -self.position.pe.size(1):]
            logits = self(context)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)
        return x

## 8. Training

### Next Token Prediction 与 target shift

保留原来的训练样本，将完整序列切分为 input 和 target：

```text
完整序列：[10, 20, 30, 40]
input：   [10, 20, 30]
target：  [20, 30, 40]
```

target 相对 input 向后移一位。位置 0 用 token 10 预测 20，位置 1 用前缀 10、20
预测 30，以此类推。遮罩确保当前位置只看到自身及过去；不做 shift 就变成预测当前 token。

logits `(B, T, V)` 展平为 `(B*T, V)`，target `(B, T)` 展平为 `(B*T,)`，
让每个位置贡献一项交叉熵。沿用 Adam、学习率 0.001 和 20 个 epoch。
这里只观察训练 loss，不包含验证集或泛化评估；两条合成序列不足以学习自然语言。

In [8]:
sequences = torch.tensor([[10, 20, 30, 40], [20, 30, 40, 50]])
x_train = sequences[:, :-1]
y_train = sequences[:, 1:]

model = MiniGPT(vocab_size=100, embed_dim=32, num_heads=4,
                hidden_dim=128, num_layers=2, max_len=20)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(20):
    logits = model(x_train)
    loss = loss_fn(logits.reshape(-1, model.lm_head.out_features), y_train.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0 or epoch == 19:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 5.2902
Epoch 5, Loss: 3.9523
Epoch 10, Loss: 2.9632
Epoch 15, Loss: 2.3199
Epoch 19, Loss: 1.9540


## 9. Generation

### Autoregressive Generation

```text
输入 prompt → 取最后位置的 logits → 预测下一个 token → 拼接到序列 → 继续预测
```

保留原来的温度采样：`logits / temperature → softmax → multinomial`。
温度必须大于 0；较小的温度让分布更集中，较大的温度让分布更平坦。
每轮只采样一个 token，循环结束后返回完整序列。

`model.eval()` 切换到推理模式，`@torch.no_grad()` 关闭梯度记录。
当上下文超过 `max_len=20` 时，只将最近 20 个 token 输入模型，
位置编码在当前窗口从 0 重新计数；返回值保留 prompt 和所有新增 token。
这是一个简单的滑动窗口策略，不包含 KV cache 或 EOS 停止条件。

下面从原训练样本前缀出发生成 20 个 token。采样结果只用于观察流程，
不应将它解释为语言能力或泛化效果。

In [9]:
model.eval()
torch.manual_seed(42)
prompt = torch.tensor([[10, 20, 30]])
generated = model.generate(prompt, max_new_tokens=20, temperature=1.0)
print("prompt:", prompt.tolist())
print("generated:", generated.tolist())
print("新增 token 数:", generated.size(1) - prompt.size(1))

prompt: [[10, 20, 30]]
generated: [[10, 20, 30, 40, 90, 84, 19, 25, 42, 72, 35, 71, 40, 29, 49, 74, 98, 50, 12, 24, 7, 11, 28]]
新增 token 数: 20


至此，输入、位置编码、因果注意力、Decoder、词表输出、梯度训练和自回归采样形成闭环。
学习时可以对照第 3 节的 shape，沿 forward 逐行追踪信息如何流动。
论文链接和项目运行方式见 [README](../README.md)。